# WOS data preparation for systematic map

Load all `.xls` files from `data/wos`, standardize columns, deduplicate, drop empty abstracts, sort, compute token stats, and partition cleaned data into 10,000-row Excel chunks under `data/partitions`.


In [4]:
from pathlib import Path
import re
import json
import tiktoken
from collections import Counter

import pandas as pd
import numpy as np

load_dir = Path('data/wos')
output_dir = Path('data/partitions')
partition_size = 10_000
KEEP_COLUMNS = [
    'Publication Type',
    'Authors',
    'Article Title',
    'Language',
    'Document Type',
    'Author Keywords',
    'Keywords Plus',
    'Abstract',
    'ORCIDs',
    'Funding Orgs',
    'Publisher',
    'ISSN',
    'eISSN',
    'ISBN',
    'Journal Abbreviation',
    'Journal ISO Abbreviation',
    'Publication Date',
    'Publication Year',
    'Volume',
    'Issue',
    'Part Number',
    'DOI',
    'DOI Link',
    'WoS Categories',
    'Web of Science Index',
    'Research Areas',
    'IDS Number',
    'Pubmed Id',
    'UT (Unique WOS ID)',
]
output_dir.mkdir(parents=True, exist_ok=True)
summary = {}

print(f'Load directory: {load_dir.resolve()}')
print(f'Output directory: {output_dir.resolve()}')
print(f'Partition size: {partition_size:,} rows per file')
print(f'Keeping {len(KEEP_COLUMNS)} columns and dropping the rest')


Load directory: /Users/hunain/d/coding-projects/biodiversity/data/wos
Output directory: /Users/hunain/d/coding-projects/biodiversity/data/partitions
Partition size: 10,000 rows per file
Keeping 29 columns and dropping the rest


In [2]:
files = sorted(load_dir.rglob('*.xls'))
summary['total_files'] = len(files)
print(f'Found {len(files)} .xls files under {load_dir}')
if not files:
    raise FileNotFoundError('No .xls files were found in data/wos')

print('Sample files (first 10):')
for path in files[:10]:
    print(f' - {path}')


Found 2790 .xls files under data/wos
Sample files (first 10):
 - data/wos/2000/savedrecs (1).xls
 - data/wos/2000/savedrecs (10).xls
 - data/wos/2000/savedrecs (11).xls
 - data/wos/2000/savedrecs (12).xls
 - data/wos/2000/savedrecs (13).xls
 - data/wos/2000/savedrecs (14).xls
 - data/wos/2000/savedrecs (15).xls
 - data/wos/2000/savedrecs (16).xls
 - data/wos/2000/savedrecs (17).xls
 - data/wos/2000/savedrecs (18).xls


In [3]:
preferred_engine = 'calamine'  # fastest for .xls if installed
engines_to_try = [preferred_engine, None, 'openpyxl', 'xlrd']

def load_with_fallback(path):
    errors = []
    for engine in engines_to_try:
        kwargs = {} if engine is None else {'engine': engine}
        engine_name = engine or 'auto'
        try:
            df = pd.read_excel(path, **kwargs)
            return df, engine_name
        except Exception as exc:
            errors.append(f"{engine_name}: {exc}")
    raise RuntimeError('; '.join(errors))

loaded_frames = []
failed_files = []
column_signatures = Counter()

for path in files:
    try:
        frame, engine_used = load_with_fallback(path)
        column_signatures[tuple(frame.columns)] += 1
        frame = frame.reindex(columns=KEEP_COLUMNS)
        loaded_frames.append(frame)
    except Exception as exc:
        failed_files.append((path, str(exc)))

summary['loaded_files'] = len(loaded_frames)
summary['failed_files'] = len(failed_files)

print(f"Files attempted: {len(files)} | Loaded: {summary['loaded_files']} | Failed: {summary['failed_files']}")
if failed_files:
    print('Failed files (path and last error):')
    for path, err in failed_files:
        print(f' - {path}: {err}')


Files attempted: 2790 | Loaded: 2790 | Failed: 0


In [4]:
if not loaded_frames:
    raise RuntimeError('No data frames were loaded; please resolve the load errors above.')

print(f'Unique column signatures (ordered): {len(column_signatures)}')
for cols, count in column_signatures.most_common():
    print(f' - {count} file(s): {len(cols)} columns')
    print(f"   Columns: {', '.join(cols)}")

most_common_cols = list(column_signatures.most_common(1)[0][0])
if len(column_signatures) > 1:
    print('Column differences relative to the most common signature:')
    base_set = set(most_common_cols)
    for cols, count in column_signatures.items():
        col_set = set(cols)
        missing = sorted(base_set - col_set)
        extra = sorted(col_set - base_set)
        if missing or extra:
            print(f" - {count} file(s): missing={missing or 'None'}, extra={extra or 'None'}")
else:
    print('All files share the same column order.')

aligned_frames = [frame.reindex(columns=KEEP_COLUMNS) for frame in loaded_frames]
print(f'Aligned all frames to {len(KEEP_COLUMNS)} kept columns.')


Unique column signatures (ordered): 1
 - 2790 file(s): 72 columns
   Columns: Publication Type, Authors, Book Authors, Book Editors, Book Group Authors, Author Full Names, Book Author Full Names, Group Authors, Article Title, Source Title, Book Series Title, Book Series Subtitle, Language, Document Type, Conference Title, Conference Date, Conference Location, Conference Sponsor, Conference Host, Author Keywords, Keywords Plus, Abstract, Addresses, Affiliations, Reprint Addresses, Email Addresses, Researcher Ids, ORCIDs, Funding Orgs, Funding Name Preferred, Funding Text, Cited References, Cited Reference Count, Times Cited, WoS Core, Times Cited, All Databases, 180 Day Usage Count, Since 2013 Usage Count, Publisher, Publisher City, Publisher Address, ISSN, eISSN, ISBN, Journal Abbreviation, Journal ISO Abbreviation, Publication Date, Publication Year, Volume, Issue, Part Number, Supplement, Special Issue, Meeting Abstract, Start Page, End Page, Article Number, DOI, DOI Link, Book DOI, 

In [5]:
combined = pd.concat(aligned_frames, ignore_index=True)
required_columns = ['Article Title', 'Abstract', 'Publication Year', 'UT (Unique WOS ID)']
missing_required = [col for col in required_columns if col not in combined.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

summary['rows_before_dedup'] = len(combined)
duplicate_rows = int(combined.duplicated(subset=['UT (Unique WOS ID)']).sum())
summary['duplicate_rows'] = duplicate_rows
print(f"Combined rows (before dedup): {summary['rows_before_dedup']:,}")
print(f"Duplicate rows detected (by UT): {duplicate_rows:,}")

combined = combined.drop_duplicates(subset=['UT (Unique WOS ID)']).reset_index(drop=True)
summary['rows_after_dedup'] = len(combined)
print(f"Rows after removing UT-duplicates: {summary['rows_after_dedup']:,}")


Combined rows (before dedup): 2,776,414
Duplicate rows detected (by UT): 426,486
Rows after removing UT-duplicates: 2,349,928


In [6]:
abstract_series = combined['Abstract']
abstract_missing = abstract_series.isna() | abstract_series.astype(str).str.strip().eq('')
abstract_missing_count = int(abstract_missing.sum())
summary['abstract_null_or_empty'] = abstract_missing_count
print(f"Rows with null/empty abstracts: {abstract_missing_count:,}")

combined = combined.loc[~abstract_missing].copy()
summary['rows_after_abstract_filter'] = len(combined)
print(f"Rows after dropping null/empty abstracts: {summary['rows_after_abstract_filter']:,}")


Rows with null/empty abstracts: 12,557
Rows after dropping null/empty abstracts: 2,337,371


In [7]:
combined['Publication Year'] = pd.to_numeric(combined['Publication Year'], errors='coerce')
publication_year_missing = int(combined['Publication Year'].isna().sum())
summary['publication_year_missing'] = publication_year_missing
combined['UT (Unique WOS ID)'] = combined['UT (Unique WOS ID)'].fillna('').astype(str)
combined = combined.sort_values(by=['Publication Year', 'UT (Unique WOS ID)'], ascending=[False, True]).reset_index(drop=True)
summary['final_rows'] = len(combined)
print(f"Rows ready for export: {summary['final_rows']:,}")
print(f"Rows with missing publication year (sorted to bottom): {publication_year_missing:,}")


Rows ready for export: 2,337,371
Rows with missing publication year (sorted to bottom): 0


In [ ]:
summary_table = pd.DataFrame([
    ('Total Excel files', summary.get('total_files', 0)),
    ('Files loaded', summary.get('loaded_files', 0)),
    ('Files failed', summary.get('failed_files', 0)),
    ('Rows before dedup', summary.get('rows_before_dedup', 0)),
    ('Duplicate rows', summary.get('duplicate_rows', 0)),
    ('Rows after dedup', summary.get('rows_after_dedup', 0)),
    ('Abstract null/empty', summary.get('abstract_null_or_empty', 0)),
    ('Rows after abstract filter', summary.get('rows_after_abstract_filter', 0)),
    ('Publication year missing', summary.get('publication_year_missing', 0)),
    ('Final rows', summary.get('final_rows', 0)),
])
summary_table.columns = ['Metric', 'Count']
summary_table


In [ ]:
encoding = tiktoken.get_encoding('cl100k_base')
encode = encoding.encode_ordinary
text_series = combined[['Article Title', 'Abstract']].fillna('').astype(str).agg(' '.join, axis=1)
token_counts = pd.Series(np.fromiter((len(encode(t)) for t in text_series), dtype=np.int64))
token_stats = {
    'sum_tokens': int(token_counts.sum()),
    'avg_per_row': float(token_counts.mean()),
    'median_per_row': float(token_counts.median()),
    'percentile_25': float(token_counts.quantile(0.25)),
    'percentile_75': float(token_counts.quantile(0.75)),
}
token_stats_df = pd.DataFrame(list(token_stats.items()), columns=['Metric', 'Value'])
token_stats_df


In [ ]:
full_export_path = output_dir / 'wos_all.csv'
combined.to_csv(full_export_path, index=False)
print(f"Saved full combined dataset ({len(combined):,} rows) -> {full_export_path}")


In [ ]:
if combined.empty:
    print('No rows to partition; nothing written.')
else:
    num_partitions = int(np.ceil(len(combined) / partition_size))
    pad_width = max(2, len(str(num_partitions)))
    saved_files = []
    partition_meta = []
    for idx in range(num_partitions):
        start = idx * partition_size
        end = start + partition_size
        chunk = combined.iloc[start:end]
        _idx= str(idx + 1)
        year_series = pd.to_numeric(chunk['Publication Year'], errors='coerce')
        year_valid = year_series.dropna()
        min_year = int(year_valid.min()) if not year_valid.empty else 'na'
        max_year = int(year_valid.max()) if not year_valid.empty else 'na'
        fname = f"wos_{str(idx + 1).zfill(pad_width)}_{max_year}-{min_year}.xlsx"
        filename = output_dir / _idx/ fname
        subdir = output_dir / str(_idx)
        subdir.mkdir(parents=True, exist_ok=True)
        chunk.to_excel(filename, index=False)
        chunk_token_sum = int(token_counts.iloc[start:end].sum())
        partition_meta.append({
            'partition': idx + 1,
            'file': str(filename),
            'rows': int(len(chunk)),
            'token_sum': chunk_token_sum,
            'min_year': min_year,
            'max_year': max_year,
        })
        saved_files.append(filename)
        print(f"Saved partition {idx + 1} with {len(chunk):,} rows -> {filename} | tokens: {chunk_token_sum:,}")
    metadata_path = output_dir / 'partition_token_metadata.json'
    metadata_path.write_text(json.dumps(partition_meta, indent=2))
    print(f"Completed writing {len(saved_files)} partition file(s).")
    print(f"Partition token metadata written to {metadata_path}")


In [1]:
import pandas as pd

In [2]:
_df = pd.read_csv('data/partitions/wos_all.csv')

/var/folders/sf/ksfgwr354pl2gjdb_f0yn3vr0000gn/T/ipykernel_9819/1503801930.py:1: DtypeWarning: Columns (13,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  _df = pd.read_csv('data/partitions/wos_all.csv')


In [3]:
len(_df)

2337371

In [11]:
_df.head()

,Publication Type,Authors,Article Title,Language,Document Type,Author Keywords,Keywords Plus,Abstract,ORCIDs,Funding Orgs,...,Issue,Part Number,DOI,DOI Link,WoS Categories,Web of Science Index,Research Areas,IDS Number,Pubmed Id,UT (Unique WOS ID)
0,J,"Hughes, P",Regional intermediary actors and professionali...,English,Article,Community-led housing; collaborative housing; ...,LAND TRUSTS; SUSTAINABILITY; TRANSITIONS; WORK...,Community-led housing initiatives address hous...,"Hughes, Philippa/0009-0000-6285-8983",ESRC [ES/Y008138/1]; ESRC studentship at the U...,...,1,NaN,10.1080/02673037.2024.2439455,0.0,Environmental Studies; Regional & Urban Planni...,Social Science Citation Index (SSCI),Environmental Sciences & Ecology; Public Admin...,Q5305,NaN,WOS:001385022700001
1,J,"Koca, HU; Koca, SB; Jawad, LA",The developmental changes in the otolith morph...,English,Article,inter- and intraspecific variation; morphology...,INNER-EAR; TELEOSTEI CYPRINODONTIDAE; SAGITTA ...,Otoliths from four species of scorpaenids-Heli...,"Jawad, Laith/0000-0002-8294-2944;",Isparta University of Applied Sciences Scienti...,...,1,NaN,10.1111/azo.12532,0.0,Anatomy & Morphology; Zoology,Science Citation Index Expanded (SCI-EXPANDED),Anatomy & Morphology; Zoology,L6949,NaN,WOS:001389849900001
2,J,"Güclü, SS; Jawad, LA; Koca, HU; Cilbiz, M; Ayd...",Skeletal deformities in Sparus aurata from Tür...,English,Article,environment; mammography; sea bream; skeletal ...,SADDLEBACK-SYNDROME; OSTEOLOGICAL DEVELOPMENT;...,The first record of the abnormalities ankylosi...,"AYDIN, Celalettin/0000-0001-8993-6013; Jawad, ...",Scientific and Technological Research Council ...,...,1,NaN,10.1111/azo.12538,0.0,Anatomy & Morphology; Zoology,Science Citation Index Expanded (SCI-EXPANDED),Anatomy & Morphology; Zoology,L6949,NaN,WOS:001410940900001
3,J,"Alia, Z; Khechekhouche, E; Amara, DG; Messaoud...","Exploring diversity, abundance and ecological ...",English,Article,abundance variation; agriculture; diversity; i...,DAMAGE,The following subject provides an in-depth loo...,"Alia, Zeid/0000-0003-3184-9456; Sawicka, Barba...","King Saud University, Riyadh, Saudi Arabia; [R...",...,1.0,NaN,10.1111/azo.12539,0.0,Anatomy & Morphology; Zoology,Science Citation Index Expanded (SCI-EXPANDED),Anatomy & Morphology; Zoology,L6949,NaN,WOS:001416324100001
4,J,"Khan, S; Jawad, LA; Khan, MA; Park, JM; Bano, ...",The relationships between fish length and otol...,English,Article,Channidae; fish length; linear regression; mor...,BODY LENGTH; VALENCIENNES; MESSINA; STRAIT; GR...,The relationship between the total length of f...,"Jawad, Laith/0000-0002-8294-2944;","Council of Scientific and Industrial Research,...",...,1,NaN,10.1111/azo.12540,0.0,Anatomy & Morphology; Zoology,Science Citation Index Expanded (SCI-EXPANDED),Anatomy & Morphology; Zoology,L6949,NaN,WOS:001427492700001


In [10]:
_df.columns

Index(['Publication Type', 'Authors', 'Article Title', 'Language',
       'Document Type', 'Author Keywords', 'Keywords Plus', 'Abstract',
       'ORCIDs', 'Funding Orgs', 'Publisher', 'ISSN', 'eISSN', 'ISBN',
       'Journal Abbreviation', 'Journal ISO Abbreviation', 'Publication Date',
       'Publication Year', 'Volume', 'Issue', 'Part Number', 'DOI', 'DOI Link',
       'WoS Categories', 'Web of Science Index', 'Research Areas',
       'IDS Number', 'Pubmed Id', 'UT (Unique WOS ID)'],
      dtype='object')

In [ ]:
_df['Date of Export'].min()

In [4]:
pip install osfclient

  Using cached osfclient-0.0.5-py2.py3-none-any.whl.metadata (5.5 kB)
Using cached osfclient-0.0.5-py2.py3-none-any.whl (39 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from osfclient.api import OSF

def upload_to_osf_direct():
    pat = input("Enter your OSF PAT: ").strip()
    project_id = input("Enter your OSF Project ID: ").strip()
    local_path = input("Enter local file/folder path: ").strip()

    # Initialize OSF client
    osf = OSF(token=pat)
    project = osf.project(project_id)
    storage = project.storage('osfstorage')

    if os.path.isdir(local_path):
        # Upload folder
        for root, dirs, files in os.walk(local_path):
            for file in files:
                full_path = os.path.join(root, file)
                # Create remote path relative to the input folder
                rel_path = os.path.relpath(full_path, os.path.dirname(local_path))
                with open(full_path, 'rb') as f:
                    storage.create_file(rel_path, f, force=True)
                    print(f"Uploaded: {rel_path}")
    else:
        # Upload single file
        file_name = os.path.basename(local_path)
        with open(local_path, 'rb') as f:
            storage.create_file(file_name, f, force=True)
            print(f"Uploaded: {file_name}")

# if __name__ == "__main__":
#     upload_to_osf_direct()


In [6]:
# foigbkt1LxxgygNA5Cs6OZlAq3jnBuYDvVCRUl14LX2zCpF19bW3GrjGh6gCBhtxbebzJB
# project id = sna2g
# file = data/partitions/wos_all.csv
upload_to_osf()

Starting upload of '/Users/hunain/d/coding-projects/biodiversity/data/partitions/wos_all.csv' to OSF Project sna2g...
Upload successful!
